# Reproducing the Math Correctness Blog Post Generation

Generate synthetic financial PDF documents (P&L / Income Statements) with
mathematically consistent numbers, following the workflow from the
[Benchmarking Coding Agents as Math Auditors](https://www.dataframer.ai/posts/benchmarking-coding-agents-as-math-auditors) blog post.

In [ ]:
%pip install dataframer pyyaml --quiet

In [ ]:
import os
import time
import zipfile
import tempfile
import urllib.request
from pathlib import Path

import yaml
from dataframer import Dataframer

client = Dataframer()

# Download seed PDFs from GitHub
GITHUB_RAW = "https://raw.githubusercontent.com/aimonlabs/dataframer-docs-public/main/generate_financial_pdfs_seeds"
SEED_FILENAMES = ["generated_sample_1.pdf", "generated_sample_2.pdf", "generated_sample_3.pdf"]

SEEDS_DIR = Path(tempfile.mkdtemp(prefix="financial_pdf_seeds_"))
for fname in SEED_FILENAMES:
    url = f"{GITHUB_RAW}/{fname}"
    dest = SEEDS_DIR / fname
    urllib.request.urlretrieve(url, dest)

seed_files = sorted(SEEDS_DIR.glob("*.pdf"))
print(f"Downloaded {len(seed_files)} seed PDFs to {SEEDS_DIR}: {[f.name for f in seed_files]}")

## 1. Upload seed dataset

In [ ]:
DATASET_NAME = "math-correctness-blog-seeds"
SPEC_NAME = "math-correctness-blog-spec"

# Clean up existing spec and dataset with same names (so the notebook is re-runnable)
for s in client.dataframer.specs.list():
    if s.name == SPEC_NAME:
        print(f"Deleting existing spec: {s.id} (status: {s.status})")
        client.dataframer.specs.delete(s.id, force=True)

for ds in client.dataframer.seed_datasets.list():
    if ds.name == DATASET_NAME:
        print(f"Deleting existing dataset: {ds.id}")
        client.dataframer.seed_datasets.delete(ds.id)

In [ ]:
# Zip the seed PDFs and upload as a MULTI_FILE dataset
with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as tmp:
    zip_path = tmp.name
    with zipfile.ZipFile(tmp, "w") as zf:
        for pdf in seed_files:
            zf.write(pdf, pdf.name)

print(f"Created zip at {zip_path}")

with open(zip_path, "rb") as f:
    dataset = client.dataframer.seed_datasets.create_from_zip(
        name=DATASET_NAME,
        zip_file=f,
    )

os.unlink(zip_path)
dataset_id = dataset.id
print(f"Dataset created: {dataset_id}")
print(f"Type: {dataset.dataset_type}, Files: {dataset.file_count}")

## 2. Create spec from seed data

In [ ]:
spec_response = client.dataframer.specs.create(
    name=SPEC_NAME,
    dataset_id=dataset_id,
    generate_distributions=True,
    generate_conditional_distributions=True,
)
spec_id = spec_response.id
print(f"Spec creation started: {spec_id}")

In [ ]:
# Poll until spec is ready
print("Waiting 30s before first poll...")
time.sleep(30)

poll_interval = 20
elapsed = 30
while True:
    spec = client.dataframer.specs.retrieve(spec_id)
    print(f"[{elapsed}s] Status: {spec.status}")
    if spec.status in ("SUCCEEDED", "FAILED"):
        break
    time.sleep(poll_interval)
    elapsed += poll_interval
    if elapsed > 120:
        poll_interval = 30
    if elapsed > 300:
        poll_interval = 60

print(f"\nSpec final status: {spec.status}")

## 3. Review and edit the spec

In [ ]:
# Display the spec YAML
spec_yaml_str = spec.content_yaml
print(spec_yaml_str)

In [ ]:
# Append calculator-tool mandate to requirements if not already present
CALCULATOR_MANDATE = """\n\nVery important: the document must be 100% internally consistent. All generated numbers must be consistent with each other. You MUST use the calculator tool to verify every single numerical relationship in the document for every number.

Before using the calculator, first plan the complete structure of the document: every table (if it should have tables of course), every row label, every column, every entity/location. Decide what line items exist and how they roll up into subtotals and totals. Only after that create SINGLE, COMPLETE script with ALL calculations included in that one script, using the calculator to verify/derive or compute every single number that will appear in the document. Specifically:
- If there are tables: this applies across ALL columns and ALL rows of ALL tables, everything must be verified in ONE script.
- You are not allowed to rely solely on mental arithmetic for any calculation. If a number is to appear in the document, the script must confirm or compute it.
- If the document contains multiple entities (locations, departments, or time periods as an example, but also any other entities), the calculator must compute numbers for ALL of them — not just one as a template.
- A couple of examples of required calculations: Every subtotal and total must be verified as the exact sum of its component line items. Every percentage must be verified as the correct division, rounded to the displayed precision. Every cross-entity consolidation (e.g. summing locations, departments, or facilities) must be verified for every row.

There is NO tolerance for even small errors in arithmetic operations — 0. This means if a sum or percentage or any aggregate deviates from its correct value even by $0.01, this is a grave problem that cannot remain in the final document.

If you need to recompute some things even after the script execution, call the calculator again for those things. If you are already satisfied with execution results, do not call the calculator again — use what you got."""

spec_yaml = yaml.safe_load(spec_yaml_str)
requirements = spec_yaml["spec"].get("requirements", "")

if "calculator tool" not in requirements.lower():
    print("Calculator mandate NOT found in requirements — appending it.")
    spec_yaml["spec"]["requirements"] = requirements + CALCULATOR_MANDATE
    updated_yaml_str = yaml.dump(spec_yaml, default_flow_style=False, allow_unicode=True, width=120)
    updated_spec = client.dataframer.specs.update(spec_id, content_yaml=updated_yaml_str)
    print(f"Spec updated: {updated_spec.id}")
else:
    print("Calculator mandate already present in requirements — no update needed.")

In [ ]:
# Show the final requirements
final_spec = client.dataframer.specs.retrieve(spec_id)
final_yaml = yaml.safe_load(final_spec.content_yaml)
print("=== REQUIREMENTS ===")
print(final_yaml["spec"]["requirements"])
print("\n=== PROPERTIES ===")
for prop in final_yaml["spec"].get("data_property_variations", []):
    print(f"\n{prop['property_name']}:")
    for val in prop.get("property_values", []):
        dist = prop.get("base_distributions", {}).get(val, "?")
        print(f"  {val}: {dist}%")

## 4. Start generation run

In [ ]:
# PAUSE: Review the spec above before running this cell.

OPUS_THINKING = "anthropic/claude-opus-4-6-thinking"
THINKING_BUDGET = 32768

run = client.dataframer.runs.create(
    spec_id=spec_id,
    number_of_samples=2,
    generation_model=OPUS_THINKING,
    generation_thinking_budget=THINKING_BUDGET,
    revision_model=OPUS_THINKING,
    revision_thinking_budget=THINKING_BUDGET,
    revision_types=["conformance"],
    max_revision_cycles=1,
    filtering_types=["conformance"],
    skip_outline=True,
    tools=["calculator"],
)

run_id = run.id
print(f"Run started: {run_id}")

In [ ]:
# Poll until generation completes
print("Waiting 60s before first poll...")
time.sleep(60)

poll_interval = 20
elapsed = 60
while True:
    run_status = client.dataframer.runs.retrieve(run_id)
    print(f"[{elapsed}s] Status: {run_status.status} | Completed: {run_status.samples_completed} | Failed: {run_status.samples_failed}")
    if run_status.status in ("SUCCEEDED", "FAILED"):
        break
    time.sleep(poll_interval)
    elapsed += poll_interval
    if elapsed > 300:
        poll_interval = 60

print(f"\nRun final status: {run_status.status}")

In [ ]:
# Download generated files
output_dir = Path(tempfile.mkdtemp(prefix="financial_pdf_generated_"))

while True:
    dl_response = client.dataframer.runs.files.download_all(run_id)
    if dl_response.download_url:
        break
    print("ZIP still being prepared, retrying in 5s...")
    time.sleep(5)

print(f"Download URL ready (size: {dl_response.size_bytes} bytes)")

zip_output = output_dir / "generated_files.zip"
urllib.request.urlretrieve(dl_response.download_url, zip_output)
print(f"Downloaded to {zip_output}")

with zipfile.ZipFile(zip_output, "r") as zf:
    zf.extractall(output_dir)
    print(f"Extracted {len(zf.namelist())} files to {output_dir}: {zf.namelist()}")